# core

> the vault, and the one small base every table here is built on

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Four tables live in the vault: the queue, the lists, the notes and the schedules. All four are
made on first use, store their structured columns as JSON, and read back as `AttrDict`. `Store`
holds that. Each of the four is then a small class instead of a fourth copy of the same twenty
lines.

The transaction comes from litesearch, not from here. `write_txn` takes the write lock upfront
with `BEGIN IMMEDIATE`. `db_lock` keys its re-entrant lock on the connection. Hand this an open
vault and an index write and a queue write take that same lock. A document write can then commit
with the job it caused.

In [ ]:
#| export
import time, uuid
from fastcore.all import AttrDict, L, first, ifnone, loads, patch, store_attr, xdumps
from litesearch import BUSY_TIMEOUT_MS, database, write_txn

In [ ]:
#| export
MAX_TEXT = 20_000    # chars kept of any one body: a model's answer is a note, not a document
JSON_COLS = ('payload', 'meta', 'result', 'items')   # columns every table here stores as JSON

## Opening the vault

`vault` is the standalone route, for a cron entry that has no reason to load a search index.
`sem_search=False` skips the usearch extensions, so a tick that finds nothing due costs one
indexed read. Where something already has the vault open, pass that database in instead: every
class here takes one rather than opening its own.

In [ ]:
#| export
def vault(path=None,                      # the file; in memory when None
          busy_ms:int=BUSY_TIMEOUT_MS     # lock wait, because a search index holds the file for longer
):
    "Open the vault for scheduling alone: no usearch extensions, so a tick pays nothing for search."
    return database(':memory:' if path is None else str(path), sem_search=False, busy_timeout=busy_ms)

In [ ]:
#| export
def now() -> float:
    'Seconds since the epoch. Every call here takes `at=None` or `now=None` and fills it from this.'
    return time.time()

def mkid() -> str:
    'A short id, the width every table here uses.'
    return uuid.uuid4().hex[:12]

def clip(s, mx:int=MAX_TEXT) -> str:
    'Text short enough to store, saying what it dropped rather than losing it quietly.'
    s = '' if s is None else str(s)
    return s if len(s) <= mx else s[:mx] + f'\n... [{len(s)-mx} more chars]'

In [ ]:
test_eq(clip('short'), 'short')
test_eq(clip('abcdef', 3), 'abc\n... [3 more chars]')
test_eq(clip(None), '')

## Rows

`xdumps` on the way in and `loads` on the way out, both from fastcore. `xdumps(None)` is the
string `null` and `loads('null')` is `None`, so a column that was never set and a column set to
nothing read back the same. Nothing here needs a JSON helper of its own.

In [ ]:
#| export
def row(r) -> AttrDict:
    'One row with its JSON columns parsed, or None.'
    if r is None: return None
    d = dict(r)
    for k in JSON_COLS:
        # `loads` answers `{}` for an empty string, and an unset column is not an empty dict
        if isinstance(d.get(k), str): d[k] = loads(d[k]) if d[k] else None
    return AttrDict(d)

def rows(rs) -> L:
    'Every row, parsed.'
    return L(rs).map(row)

In [ ]:
test_eq(row(dict(id='a', payload=xdumps({'qty': 2}))).payload, {'qty': 2})
test_eq(row(dict(id='a', payload=xdumps(None))).payload, None)   # `null` round trips to None
test_eq(row(dict(id='a', payload=None)).payload, None)           # and so does a SQL NULL
test_eq(row(None), None)

## Store

In [ ]:
#| export
class Store:
    """One table in the vault, made on first use and read back as parsed rows.

    A subclass sets `name`, `cols` and `idx`, and gets construction, the busy timeout and row
    parsing. That is why the queue, the lists, the notes and the schedules are four small classes.
    """
    name:str = ''    # the table
    cols:dict = {}   # column to type, as fastlite's `Table.create` takes them
    pk:str = 'id'
    idx:tuple = ()   # `CREATE INDEX IF NOT EXISTS` statements, run on every open

    def __init__(self,
                 db,                            # an open vault: `vault()`, or one litesearch opened
                 busy_ms:int=BUSY_TIMEOUT_MS):  # lock wait before a write gives up
        store_attr()
        # a search index writing chunks holds this file far longer than apsw's 100ms default
        db.conn.set_busy_timeout(busy_ms)
        self.t = getattr(db.t, self.name)
        self.t.create(**self.cols, pk=self.pk, if_not_exists=True)
        for sql in self.idx: db.conn.execute(sql)

    def q(self, sql:str, args=()) -> L:
        'Every row `sql` returns, parsed.'
        return rows(self.db.q(sql, list(args)))

    def one(self, sql:str, args=()) -> AttrDict:
        'The first row `sql` returns, or None.'
        return row(first(self.db.q(sql, list(args))))

    def txn(self):
        'The write transaction, so no subclass imports litesearch itself.'
        return write_txn(self.db)

A subclass is the three class attributes and nothing else:

In [ ]:
class Demo(Store):
    "Two columns and an index, to show what a subclass costs."
    name = 'demo'
    cols = dict(id=str, topic=str, payload=str, at=float)
    idx = ('CREATE INDEX IF NOT EXISTS demo_topic ON demo(topic, at)',)

db = vault()
d = Demo(db)
with d.txn(): d.t.insert(dict(id=mkid(), topic='milk', payload=xdumps({'qty': 2}), at=now()))
got = d.one('SELECT * FROM demo WHERE topic=?', ['milk'])
test_eq(got.payload, {'qty': 2})   # parsed coming out, not a string
test_eq(got.topic, 'milk')

The reason the queue shares the vault rather than keeping a file of its own: a write that fails
takes the row it caused with it.

In [ ]:
try:
    with d.txn():
        d.t.insert(dict(id=mkid(), topic='eggs', payload=None, at=now()))
        raise RuntimeError('the document write failed')
except RuntimeError: pass
test_eq(len(d.q('SELECT * FROM demo WHERE topic=?', ['eggs'])), 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()